<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/05_medical_baseline_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tunix-Med: Baseline Medical Knowledge Evaluation

Measures how well the **pre-trained `google/gemma-3-1b-it` model** answers
cardiology questions **before any fine-tuning**.  The scores produced here
are the **baseline** to compare against after Tunix domain-specific training.

### Evaluation pipeline

For each question the notebook:
1. Generates an answer using the pre-trained model (no adapter loaded).
2. Computes **perplexity** of the reference answer.
3. Scores the answer with three independent methods:
   - **Keyword matching**
   - **Semantic similarity** (Sentence-BERT `all-MiniLM-L6-v2`)
   - **AI judge** (Qwen2.5-7B-Instruct in 4-bit)
4. Saves results to `medical_baseline_results.csv` for comparison.

> **Important:** The questions are sampled from the held-out split of
> `lmassaron/medical-cardiology-qa` (seed=42, 10% eval) — the **same 25
> questions** used in notebook 08, so the two sets of
> scores are directly comparable.

## 1 · Setup & Model Loading

In [1]:
import os, warnings, re
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import login

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)


def info_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


device = info_device()
# Gemma 3 overflows float16; use bfloat16 on Ampere+ GPUs, float32 otherwise.
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float32
)
print(f"Device : {device}  |  dtype : {dtype}")

MODEL_NAME = "google/gemma-3-1b-it"
print("Loading base model (no adapter) ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,  # correct kwarg for current Transformers
    device_map=device,
)
model.eval()
print("Model ready.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device : cuda  |  dtype : torch.bfloat16
Loading base model (no adapter) ...
Model ready.


## 2 · Evaluation Dataset

We load the held-out questions from `lmassaron/medical-cardiology-qa` using
the **identical** seed, split fraction, and sampling logic as notebook 08.  This guarantees the baseline and fine-tuned scores are
measured on exactly the same questions.

In [2]:
DATASET_ID = "lmassaron/medical-cardiology-qa"
EVAL_SPLIT = 0.1
SEED = 42
N_EVAL_QS = 300

print(f"Loading {DATASET_ID} ...")
full_ds = load_dataset(DATASET_ID, split="train")
n = len(full_ds)

rng = np.random.default_rng(SEED)
all_idx = rng.permutation(n)
cut = int(n * (1.0 - EVAL_SPLIT))
eval_idx = all_idx[cut:]

print(f"  Total rows : {n:,}  |  Eval rows : {len(eval_idx):,}")


def extract_qa(example):
    msgs = example["messages"]
    return {
        "question": next(m["content"] for m in msgs if m["role"] == "user"),
        "answer": next(m["content"] for m in msgs if m["role"] == "assistant"),
    }


rng2 = np.random.default_rng(SEED + 1)
shuffled = rng2.permutation(eval_idx)

seen_prefixes, qa_pairs = set(), []
for idx in shuffled:
    if len(qa_pairs) >= N_EVAL_QS:
        break
    ex = extract_qa(full_ds[int(idx)])
    q, a = ex["question"], ex["answer"]
    if len(a) < 25:
        continue
    prefix = " ".join(q.lower().split()[:4])
    if prefix in seen_prefixes:
        continue
    seen_prefixes.add(prefix)
    qa_pairs.append({"question": q, "answer": a, "dataset_idx": int(idx)})

data = pd.DataFrame(qa_pairs)
print(f"\nSampled {len(data)} evaluation questions")
print("\nSample questions:")
for _, row in data.head(5).iterrows():
    print(f"  Q: {row['question'][:90]}")
    print(f"  A: {row['answer'][:80]}")
    print()

Loading lmassaron/medical-cardiology-qa ...
  Total rows : 10,518  |  Eval rows : 1,052

Sampled 300 evaluation questions

Sample questions:
  Q: What is the rate of bleeding stroke associated with statin use over 5 years of treatment p
  A: Over 5 years of treatment, statins result in 7.5 cases of bleeding stroke per 10

  Q: Which medications used in cardiac stress testing can cause mild hypotension?
  A: Adenosine and dipyridamole.

  Q: What is the advantage of using perfusion stress test with 99mTc labelled sestamibi?
  A: It is appropriate for select patients with an abnormal resting electrocardiogram

  Q: What is the suggested course of action if angina is suspected?
  A: An urgent medical assessment is suggested to rule out serious medical conditions

  Q: What is the benefit of using beta blockers in glaucoma treatment?
  A: They can lower intraocular pressure.



## 3 · Baseline Inference Loop

We use the **same system prompt** that was used during fine-tuning so the
prompt format is identical between baseline and fine-tuned evaluations.
Without this, the comparison is unfair — a different prompt shifts the output
distribution even before any training effect.

In [3]:
# Must match the system prompt in the training data exactly
SYSTEM_PROMPT = (
    "You are a knowledgeable medical assistant specializing in cardiology. "
    "Answer clinical questions accurately, focusing on diagnostic criteria, "
    "treatment guidelines, and pathophysiology."
)

results_list = []
model.eval()

for _, row in tqdm(data.iterrows(), total=len(data)):
    question = row["question"]
    answer = row["answer"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    input_ids = encoded.to(device)
    attention_mask = torch.ones_like(
        input_ids
    )  # explicit mask avoids pad-token ambiguity

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    gen_ids = outputs[0, input_ids.shape[-1] :]
    generated = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    # Reference perplexity: how surprised the *base* model is by the correct answer
    ref_ids = tokenizer(
        answer, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_ids = torch.cat([input_ids, ref_ids], dim=1)
    labels = full_ids.clone()
    labels[:, : input_ids.shape[1]] = -100
    with torch.no_grad():
        loss = model(
            full_ids, attention_mask=torch.ones_like(full_ids), labels=labels
        ).loss
    perplexity = torch.exp(loss).item()

    results_list.append(
        {
            "question": question,
            "expected_answer": answer,
            "generated_answer": generated,
            "perplexity": perplexity,
        }
    )

results_df = pd.DataFrame(results_list)
print(f"Generated {len(results_df)} answers")
results_df[["question", "expected_answer", "generated_answer", "perplexity"]].head()

100%|██████████| 300/300 [20:05<00:00,  4.02s/it]

Generated 300 answers


,question,expected_answer,generated_answer,perplexity
0,What is the rate of bleeding stroke associated...,"Over 5 years of treatment, statins result in 7...","Okay, let’s tackle this question about the rat...",27.679310
1,Which medications used in cardiac stress testi...,Adenosine and dipyridamole.,"Okay, let’s delve into the medications that ca...",2712.877197
2,What is the advantage of using perfusion stres...,It is appropriate for select patients with an ...,"Okay, let’s delve into the advantages of the P...",67170.296875
3,What is the suggested course of action if angi...,An urgent medical assessment is suggested to r...,"Okay, let’s talk about managing suspected angi...",12280.543945
4,What is the benefit of using beta blockers in ...,They can lower intraocular pressure.,"Okay, let’s delve into the fascinating and som...",774856.687500


## Scoring Metrics

Three metrics, each addressing a specific failure mode of the original scoring:

| Metric | Old problem | Fix |
|---|---|---|
| **Keyword F1** | Recall-only rewarded verbose outputs | F1 (precision+recall) with TF-IDF weights |
| **Semantic similarity** | Range compressed to [0.5, 1.0] — wrong answers scored ~0.6 | Calibrated to empirical [min, max] of the eval set |
| **AI judge** | No verbosity penalty; committed to score before reasoning | Explicit factual rubric; chain-of-thought before final score |

In [4]:
# ════════════════════════════════════════════════════════════════════════════
# Metric 1 — Keyword F1 with TF-IDF weighting
# ════════════════════════════════════════════════════════════════════════════
# Problems with the old metric:
#   • Pure recall (found/total) rewards verbose outputs: a model that writes
#     200 words finds more keywords by sheer volume, even if mostly wrong.
#   • All keywords treated equally — "the" and "neprilysin" scored the same.
#
# Fix:
#   • F1 of keywords = harmonic mean of precision and recall.
#     Precision = fraction of keywords in the generated answer that also
#     appear in the reference.  Recall = fraction of reference keywords
#     found in the generated answer.  A verbose wrong answer has low
#     precision; a short correct answer has low recall but high precision;
#     F1 rewards the right balance.
#   • TF-IDF weights: rare domain terms (e.g. drug names, anatomical terms)
#     contribute more than common words.  We fit a TF-IDF vectorizer over
#     the reference answers in the eval set and use its idf_ array to weight
#     each keyword's contribution.
# ════════════════════════════════════════════════════════════════════════════
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Fit TF-IDF on all reference answers so rare medical terms get higher weight
_tfidf = TfidfVectorizer(analyzer="word", token_pattern=r"\b\w{4,}\b",
                         sublinear_tf=True)
_tfidf.fit(results_df["expected_answer"].tolist())
_vocab   = _tfidf.vocabulary_          # word → column index
_idf     = _tfidf.idf_                 # column index → idf weight


def keyword_f1_tfidf(generated: str, expected: str) -> float:
    """
    TF-IDF-weighted keyword F1 between generated and reference answer.
    Returns a value in [0, 1].  Penalises verbose wrong answers.
    """
    ref_kws = set(re.findall(r"\b\w{4,}\b", expected.lower()))
    gen_kws = set(re.findall(r"\b\w{4,}\b", generated.lower()))
    if not ref_kws:
        return 1.0

    def weighted_count(kws_to_check, universe):
        total = 0.0
        for w in universe:
            if w in kws_to_check:
                total += _idf[_vocab[w]] if w in _vocab else 1.0
        return total

    ref_weight = sum(_idf[_vocab[w]] if w in _vocab else 1.0 for w in ref_kws)
    gen_weight = sum(_idf[_vocab[w]] if w in _vocab else 1.0 for w in gen_kws) if gen_kws else 0.0

    if ref_weight == 0 or gen_weight == 0:
        return 0.0

    recall    = weighted_count(gen_kws, ref_kws) / ref_weight
    precision = weighted_count(ref_kws, gen_kws) / gen_weight
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ════════════════════════════════════════════════════════════════════════════
# Metric 2 — Calibrated semantic similarity
# ════════════════════════════════════════════════════════════════════════════
# Problem: cosine similarity from sentence-BERT clusters in a narrow band
# [~0.3, ~1.0] for any pair of topically related medical sentences.
# A wrong answer scores ~0.5-0.65, which looks like "partial credit" but is
# essentially chance-level for medical text.
#
# Fix: stretch the score to the full [0, 1] range using the empirical
# minimum and maximum across all (generated, expected) pairs in the eval set.
# score_calibrated = (raw - min_raw) / (max_raw - min_raw)
# We compute raw scores first, then calibrate in one pass.
# ════════════════════════════════════════════════════════════════════════════
from sentence_transformers import SentenceTransformer, util

sim_model = SentenceTransformer("all-MiniLM-L6-v2")


def _raw_semantic(generated: str, expected: str) -> float:
    e1 = sim_model.encode(generated, convert_to_tensor=True)
    e2 = sim_model.encode(expected,  convert_to_tensor=True)
    return float(util.pytorch_cos_sim(e1, e2))


# ════════════════════════════════════════════════════════════════════════════
# Metric 3 — Improved AI judge with chain-of-thought
# ════════════════════════════════════════════════════════════════════════════
# Problems with old prompt:
#   • No verbosity penalty — long wrong answers scored as "partial"
#   • No factual specificity requirement — "sounds medical" ≠ correct
#   • Returning ONLY a number forces commitment before reasoning
#
# Fix:
#   • Prompt the judge to reason first (brief CoT), then output the score
#     on a LAST LINE by itself — we parse only that last line.
#   • Explicit rubric: factual correctness > plausibility; verbosity that
#     buries the correct answer is penalised.
#   • max_new_tokens raised to 120 to allow brief reasoning.
# ════════════════════════════════════════════════════════════════════════════
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
)
judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge_mdl = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, quantization_config=bnb_cfg, device_map="auto",
)


def ai_judge(question: str, generated: str, expected: str) -> float:
    """
    Returns a score in [0, 1].
    The judge reasons briefly then outputs an integer 1-10 on the last line.
    """
    prompt = (
        "You are a strict medical fact-checker.\n"
        "Evaluate ONLY factual correctness of the generated answer "
        "against the reference answer for the given question.\n\n"
        f"Question  : {question}\n"
        f"Reference : {expected}\n"
        f"Generated : {generated}\n\n"
        "Rules:\n"
        "- Factual accuracy is paramount. A plausible-sounding but wrong "
        "answer scores 1-3.\n"
        "- Specific facts matter: correct drug names, numbers, and mechanisms "
        "are required for a high score.\n"
        "- A correct answer that is unnecessarily verbose scores at most 7.\n"
        "- A concise, fully correct answer scores 9-10.\n"
        "- A partially correct answer (right concept, missing key detail) "
        "scores 4-6.\n\n"
        "First write ONE sentence explaining your reasoning.\n"
        "Then on the very last line write ONLY the integer score (1-10)."
    )
    inp  = judge_tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(judge_mdl.device)
    mask = torch.ones_like(inp)
    with torch.no_grad():
        out = judge_mdl.generate(
            inp, attention_mask=mask,
            max_new_tokens=120,   # enough for one reasoning sentence + score
            do_sample=False,
        )
        txt = judge_tok.decode(out[0, inp.shape[-1]:],
                               skip_special_tokens=True).strip()
    # Parse: take the last non-empty line and extract the first integer
    last_line = next((l.strip() for l in reversed(txt.splitlines()) if l.strip()), "")
    m = re.search(r"\b(\d+)\b", last_line)
    if m:
        score = int(m.group(1))
        return min(max(score, 1), 10) / 10.0
    # Fallback: search anywhere in the output
    m2 = re.search(r"\b(\d+)\b", txt)
    return min(max(int(m2.group(1)), 1), 10) / 10.0 if m2 else 0.5


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 5 · Results & Baseline Summary

In [5]:
# ── Compute raw scores ────────────────────────────────────────────────────────
print("Computing keyword F1 (TF-IDF weighted) ...")
results_df["keyword_score"] = results_df.apply(
    lambda r: keyword_f1_tfidf(r["generated_answer"], r["expected_answer"]), axis=1)

print("Computing raw semantic similarity ...")
results_df["_raw_sim"] = results_df.apply(
    lambda r: _raw_semantic(r["generated_answer"], r["expected_answer"]), axis=1)

# ── Calibrate semantic similarity to [0, 1] ───────────────────────────────────
# Stretch the empirical [min, max] to [0, 1] so the metric uses its full range.
sim_min = results_df["_raw_sim"].min()
sim_max = results_df["_raw_sim"].max()
rng = sim_max - sim_min if sim_max > sim_min else 1.0
results_df["semantic_score"] = ((results_df["_raw_sim"] - sim_min) / rng).clip(0, 1)
print(f"  Raw sim range: [{sim_min:.3f}, {sim_max:.3f}]  "
      f"→ calibrated to [0.000, 1.000]")

print("Computing AI judge scores (with chain-of-thought reasoning) ...")
results_df["ai_judge_score"] = results_df.apply(
    lambda r: ai_judge(r["question"], r["generated_answer"], r["expected_answer"]),
    axis=1)

# ── Final score ───────────────────────────────────────────────────────────────
results_df["final_score"] = (
    results_df["keyword_score"]  * 0.2 +
    results_df["semantic_score"] * 0.4 +
    results_df["ai_judge_score"] * 0.4
)

print("\n─── Results ───────────────────────────────────────────────────────")
print(f"  Keyword F1 (TF-IDF, mean)  : {results_df['keyword_score'].mean():.3f}")
print(f"  Semantic sim (calib, mean) : {results_df['semantic_score'].mean():.3f}  "
      f"(raw: {results_df['_raw_sim'].mean():.3f})")
print(f"  AI judge (CoT, mean)       : {results_df['ai_judge_score'].mean():.3f}")
print(f"  Final score (mean)         : {results_df['final_score'].mean():.3f}")
print(f"  Perplexity (mean)          : {results_df['perplexity'].mean():.1f}")
print("───────────────────────────────────────────────────────────────────")

best  = results_df.nlargest(3,  "final_score")[["question","expected_answer","generated_answer","final_score"]]
worst = results_df.nsmallest(3, "final_score")[["question","expected_answer","generated_answer","final_score"]]

print("\n── Top 3 answers ──")
for _, r in best.iterrows():
    print(f"  [{r['final_score']:.2f}]  Q: {r['question'][:70]}")
    print(f"         Expected : {r['expected_answer'][:70]}")
    print(f"         Generated: {r['generated_answer'][:70]}")
    print()

print("── Bottom 3 answers ──")
for _, r in worst.iterrows():
    print(f"  [{r['final_score']:.2f}]  Q: {r['question'][:70]}")
    print(f"         Expected : {r['expected_answer'][:70]}")
    print(f"         Generated: {r['generated_answer'][:70]}")
    print()

results_df.drop(columns=["_raw_sim"]).to_csv("results.csv", index=False)
print("Saved → results.csv")


Computing keyword F1 (TF-IDF weighted) ...
Computing raw semantic similarity ...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Raw sim range: [0.015, 0.891]  → calibrated to [0.000, 1.000]
Computing AI judge scores (with chain-of-thought reasoning) ...

─── Results ───────────────────────────────────────────────────────
  Keyword F1 (TF-IDF, mean)  : 0.102
  Semantic sim (calib, mean) : 0.572  (raw: 0.516)
  AI judge (CoT, mean)       : 0.571
  Final score (mean)         : 0.477
  Perplexity (mean)          : 30609236.4
───────────────────────────────────────────────────────────────────

── Top 3 answers ──
  [0.79]  Q: What role do social determinants play in myocardial infarction risk an
         Expected : Social determinants such as neighborhood disadvantage, immigration sta
         Generated: Okay, let’s delve into the crucial role of social determinants of heal

  [0.78]  Q: What is the difference between systolic and diastolic blood pressure?
         Expected : Systolic blood pressure is the top number representing the pressure in
         Generated: Okay, let’s dive into the differences between sys

## Conclusion

The scores above represent the **untrained capability** of `gemma-3-1b-it`
on Wikipedia-derived cardiology factual Q&A, using the same system prompt
and questions as the fine-tuned evaluation.

**What to expect after fine-tuning:**
- Lower perplexity — the model becomes less surprised by correct factual answers.
- Higher keyword and semantic scores — generated answers contain more of the
  domain-specific terms and phrasings present in the training data.
- Higher AI judge scores — responses are more factually accurate and concise.

Run `08_medical_evaluation_final.ipynb` after training to see the delta.